<a href="https://colab.research.google.com/github/manastripathi2357-alt/Acoustic-Affect-Analysis/blob/main/AAA_Summer_project_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import pickle
import wave
import collections
import contextlib
import numpy as np
import librosa
import soundfile as sf
from copy import deepcopy

# Machine Learning imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.mixture import GaussianMixture
from sklearn.cluster import SpectralClustering
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from xgboost import XGBClassifier
import kagglehub

# Fallback for VAD
try:
    import webrtcvad
except ImportError:
    webrtcvad = None

# Emotion labels mapping
EMOTION_DICT = {
    0: 'neutral', 1: 'calm', 2: 'happy', 3: 'sad',
    4: 'angry', 5: 'fearful', 6: 'disgust', 7: 'surprised'
}

print("Libraries imported and globals defined successfully!")

Libraries imported and globals defined successfully!


In [ ]:
def download_and_clean_dataset():
    """Downloads RAVDESS and filters out duplicate/Mac OS metadata files."""
    print("\n--- Downloading RAVDESS Dataset ---")
    path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
    search_pattern = os.path.join(path, '**', '*.wav')
    raw_files = glob.glob(search_pattern, recursive=True)

    unique_files = {}
    for file_path in raw_files:
        file_name = os.path.basename(file_path)
        if not file_name.startswith('._'):
            unique_files[file_name] = file_path

    audio_files = sorted(list(unique_files.values()))
    print(f"Cleaned valid audio files: {len(audio_files)}")
    return audio_files

def extract_193_features(file_path, target_sr=16000):
    """Computes the 193-dimensional feature array for a single audio file."""
    X, sr = librosa.load(file_path, sr=target_sr)
    stft = np.abs(librosa.stft(X))

    mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sr, n_mfcc=40).T, axis=0)
    chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sr).T, axis=0)
    mel = np.mean(librosa.feature.melspectrogram(y=X, sr=sr).T, axis=0)
    contrast = np.mean(librosa.feature.spectral_contrast(S=stft, sr=sr).T, axis=0)
    harmonic = librosa.effects.harmonic(X)
    tonnetz = np.mean(librosa.feature.tonnetz(y=harmonic, sr=sr).T, axis=0)

    return np.hstack([mfccs, chroma, mel, contrast, tonnetz])

def prepare_feature_data(audio_files):
    """Runs feature extraction across all files with Pickle caching."""
    feature_path = os.path.join("data", "feature.pkl")
    label_path = os.path.join("data", "label.pkl")

    if os.path.exists(feature_path) and os.path.exists(label_path):
        print("\nPre-extracted features found. Loading directly from cache...")
        with open(feature_path, "rb") as f: X_data = pickle.load(f)
        with open(label_path, "rb") as f: y_data = np.array(pickle.load(f), dtype=int)
        return X_data, y_data

    print("\nExecuting feature extraction. This requires significant computation time...")
    feature_all, labels = [], []

    # Process all files
    for file_path in audio_files:
        try:
            emotion_label = int(os.path.basename(file_path).split('-')[2])
            feature_all.append(extract_193_features(file_path))
            labels.append(emotion_label)
        except Exception:
            continue

    X_data, y_data = np.array(feature_all), np.array(labels)
    os.makedirs("data", exist_ok=True)
    with open(feature_path, "wb") as f: pickle.dump(X_data, f)
    with open(label_path, "wb") as f: pickle.dump(list(y_data), f)

    return X_data, y_data

### Calling the helper functions to download, clean, and extract features

In [ ]:
audio_files = download_and_clean_dataset()
X_data, y_data = prepare_feature_data(audio_files)

print(f"Final Feature Matrix (X_data) Shape: {X_data.shape}")
print(f"Final Labels Array (y_data) Shape: {y_data.shape}")


--- Downloading RAVDESS Dataset ---


100%|██████████| 429M/429M [00:02<00:00, 177MB/s]

Extracting files...


Cleaned valid audio files: 1440

Executing feature extraction. This requires significant computation time...
Final Feature Matrix (X_data) Shape: (1440, 193)
Final Labels Array (y_data) Shape: (1440,)


In [ ]:
class Frame(object):
    def __init__(self, bytes_data, timestamp, duration):
        self.bytes, self.timestamp, self.duration = bytes_data, timestamp, duration

def frame_generator(frame_duration_ms, audio_bytes, sample_rate):
    n = int(sample_rate * (frame_duration_ms / 1000.0) * 2)
    offset, timestamp, duration = 0, 0.0, (float(n) / sample_rate) / 2.0
    while offset + n < len(audio_bytes):
        yield Frame(audio_bytes[offset:offset + n], timestamp, duration)
        timestamp += duration; offset += n

def vad_collector(sample_rate, frame_duration_ms, padding_duration_ms, vad, frames):
    num_padding_frames = int(padding_duration_ms / frame_duration_ms)
    ring_buffer = collections.deque(maxlen=num_padding_frames)
    triggered, voiced_frames, start_time = False, [], 0.0
    for frame in frames:
        is_speech = vad.is_speech(frame.bytes, sample_rate)
        if not triggered:
            ring_buffer.append((frame, is_speech))
            if len([f for f, speech in ring_buffer if speech]) > 0.9 * ring_buffer.maxlen:
                triggered, start_time = True, ring_buffer[0][0].timestamp
                voiced_frames.extend([f for f, s in ring_buffer])
                ring_buffer.clear()
        else:
            voiced_frames.append(frame); ring_buffer.append((frame, is_speech))
            if len([f for f, speech in ring_buffer if not speech]) > 0.9 * ring_buffer.maxlen:
                triggered = False
                yield b''.join([f.bytes for f in voiced_frames]), start_time, frame.timestamp + frame.duration
                ring_buffer.clear(); voiced_frames = []
    if voiced_frames:
        yield b''.join([f.bytes for f in voiced_frames]), start_time, frames[-1].timestamp + frames[-1].duration

def MAP_Estimation(model, data, relevance_factor=16, m_iterations=1):
    for _ in range(m_iterations):
        z_n_k = model.predict_proba(data)
        n_k = np.sum(z_n_k, axis=0).reshape(-1, 1)
        mu_new = np.dot(z_n_k.T, data) / np.where(n_k == 0, 1e-20, n_k)
        adapt_coef = n_k / (n_k + relevance_factor)
        model.means_ = (adapt_coef * mu_new) + ((1.0 - adapt_coef) * model.means_)
    return model

def separate_speakers(file_path, output_dir="temp_chunks", num_speakers=2):
    print(f"\n--- Running Speaker Separation on {os.path.basename(file_path)} ---")
    os.makedirs(output_dir, exist_ok=True)
    temp_wav = os.path.join(output_dir, "temp_mono_16k.wav")

    y, sr = librosa.load(file_path, sr=16000)
    sf.write(temp_wav, y, sr, subtype='PCM_16')
    chunks = []

    # VAD
    if webrtcvad is not None:
        try:
            with contextlib.closing(wave.open(temp_wav, 'rb')) as wf:
                pcm_data = wf.readframes(wf.getnframes())
            vad = webrtcvad.Vad(2)
            frames = list(frame_generator(30, pcm_data, sr))
            for i, (seg_bytes, start, end) in enumerate(vad_collector(sr, 30, 300, vad, frames)):
                c_path = os.path.join(output_dir, f"chunk-{i:03d}.wav")
                with contextlib.closing(wave.open(c_path, 'wb')) as wf:
                    wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(sr)
                    wf.writeframes(seg_bytes)
                chunks.append({'chunk_path': c_path, 'start': start, 'end': end})
        except Exception: chunks = []

    if not chunks:
        for i, interval in enumerate(librosa.effects.split(y, top_db=30)):
            c_path = os.path.join(output_dir, f"chunk-{i:03d}.wav")
            sf.write(c_path, y[interval[0]:interval[1]], sr, subtype='PCM_16')
            chunks.append({'chunk_path': c_path, 'start': interval[0]/sr, 'end': interval[1]/sr})

    if os.path.exists(temp_wav): os.remove(temp_wav)
    if not chunks: return []

    # UBM & MAP Estimation
    h_len, n_fft, n_mfcc = 0.010, 0.032, 13
    mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=int(h_len*sr), n_fft=int(n_fft*sr), n_mfcc=n_mfcc)
    ubm_feat = np.vstack((mfcc, librosa.feature.delta(mfcc), librosa.feature.delta(mfcc, order=2))).T
    scaler = StandardScaler()
    ubm_feat = scaler.fit_transform(ubm_feat)
    ubm_model = GaussianMixture(n_components=16, covariance_type='full', reg_covar=1e-3, random_state=42).fit(ubm_feat)

    s_vecs, v_chunks = [], []
    for chunk in chunks:
        try:
            y_c, sr_c = librosa.load(chunk['chunk_path'], sr=16000)
            if len(y_c) < int(n_fft * sr_c): continue
            c_mfcc = librosa.feature.mfcc(y=y_c, sr=sr_c, hop_length=int(h_len*sr_c), n_fft=int(n_fft*sr_c), n_mfcc=n_mfcc)
            c_feat = scaler.transform(np.vstack((c_mfcc, librosa.feature.delta(c_mfcc), librosa.feature.delta(c_mfcc, order=2))).T)
            gmm = MAP_Estimation(deepcopy(ubm_model), c_feat, relevance_factor=16, m_iterations=1)
            s_vecs.append(gmm.means_.flatten()); v_chunks.append(chunk)
        except Exception: continue

    if not s_vecs: return []

    a_clusters = min(num_speakers, len(v_chunks))
    labels = SpectralClustering(n_clusters=a_clusters, affinity='cosine', random_state=42).fit_predict(np.array(s_vecs)) if a_clusters > 1 else np.zeros(len(v_chunks), dtype=int)

    seen, n_lbl, rearr = {}, 0, []
    for lbl in labels:
        if lbl not in seen: seen[lbl] = n_lbl; n_lbl += 1
        rearr.append(seen[lbl])

    for i, chunk in enumerate(v_chunks): chunk['speaker'] = rearr[i]
    return v_chunks

print("Speaker separation functions loaded!")

Speaker separation functions loaded!


In [ ]:
print("\n--- Week 4-5: Training MLP Neural Network ---")

# Standardize labels to 0-7 format for Keras
y_data_zero = y_data - np.min(y_data)
y_encoded = to_categorical(y_data_zero, num_classes=8)
X_train, X_test, y_train, y_test = train_test_split(X_data, y_encoded, test_size=0.3, random_state=42)

mlp_model = Sequential([
    Dense(400, input_dim=X_train.shape[1], kernel_initializer='normal', activation='relu'),
    Dropout(0.2),
    Dense(200, kernel_initializer='normal', activation='relu'),
    Dropout(0.2),
    Dense(100, kernel_initializer='normal', activation='relu'),
    Dropout(0.2),
    Dense(8, kernel_initializer='normal', activation='softmax')
])

mlp_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the network (you can lower epochs to 50 for a faster classroom demo)
mlp_model.fit(X_train, y_train, epochs=150, batch_size=32, validation_data=(X_test, y_test), verbose=1)

os.makedirs("models", exist_ok=True)
mlp_model.save("models/emotion_mlp_model.h5")
print("MLP Model Trained and Saved!")